<a href="https://colab.research.google.com/github/kubenko-k/Spatial-project/blob/main/Cell_types/Copy_of_Cell_types.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install scanpy


In [ ]:
pip install gseapy

In [ ]:
import warnings
import scanpy as sc
import anndata as an
import pandas as pd
import gseapy as gp
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from urllib import request
import json
import os
from tqdm.notebook import tqdm

import statsmodels.api as sm
from statsmodels.formula.api import ols
from tqdm.notebook import tqdm
from statsmodels.stats.multitest import multipletests

sc.settings.set_figure_params(dpi=80)
#sc.set_figure_params(facecolor="white", figsize=(8, 8))
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
dir_path = '/content/drive/My Drive/Skoltech/Spatial_project/'
dir_path_er = '/content/drive/My Drive/Skoltech/Spatial_project/EdgeR/'
dir_path_r = '/content/drive/My Drive/Skoltech/Spatial_project/Raw/'

In [ ]:
pb_age = sc.read_h5ad(dir_path + 'pb_age.h5ad')
pb_age.obs["lib_size"] = pb_age.X.sum(axis=1)
pb_age.obs["log_lib_size"] = np.log(pb_age.obs["lib_size"])
sc.pp.normalize_total(pb_age, target_sum=1e4)
sc.pp.log1p(pb_age)
pb_age = pb_age[:, pb_age.X.mean(axis=0) > 0.05]

In [ ]:

def mean (adata):
    pb_df = pd.DataFrame(adata.X, columns=adata.var_names, index=adata.obs_names)
    pb_df['sample_id'] = adata.obs['sample_id']
    pb_df['sample_id'].replace({'151507': '1', '151508':'1', '151509' : '1', '151510' : '1', '151669' : '2', '151670':'2', '151671':'2', '151672':'2', '151673':'3', '151674':'3', '151675':'3', '151676':'3'}, inplace=True)
    pb_df.head()
    sample_mean = pb_df.groupby('sample_id').mean()
    columns = pb_df.columns.tolist()[:-1]
    for sample in tqdm(sample_mean.index.tolist()):
        pb_df.loc[pb_df.sample_id == sample, columns] = (pb_df.loc[pb_df.sample_id == sample, columns] - sample_mean.loc[sample])

    adata.X = pb_df[columns].values
    return adata

In [ ]:
pb_age_norm_mean = mean (pb_age)

In [ ]:
edge_r = pd.read_csv(dir_path_er + 'sig_edger.csv')
pb_age_norm = sc.read_h5ad(dir_path + 'pb_age_norm.h5ad')
df = pd.DataFrame(pb_age_norm.X, index=pb_age_norm.obs_names, columns=pb_age_norm.var_names)
df_mean = pd.DataFrame(pb_age_norm_mean.X, index=pb_age_norm_mean.obs_names, columns=pb_age_norm_mean.var_names)
df_mean['condition'] = pb_age_norm_mean.obs.condition
df_mean['layer'] = pb_age_norm_mean.obs.layer
df_mean = df_mean.T
df_mean
df = df.T
df_res = df[df.index.isin(edge_r.gene)]
df_res = df_res.T
df_res = df_mean[df_mean.index.isin(edge_r.gene)]
df_res = df_res.T
df_res

In [ ]:
df_res = df_res.apply(pd.to_numeric)
sample_order = dict()
for cond in ['human', '151']:
    samples = df_res.loc[df_res.index.str.contains(cond)].index
    order = samples.sort_values()
    order_laminar = order[order.str.contains('L')].tolist()
    order_wm = order[order.str.contains('WM')].tolist()
    sample_order[cond] = order_laminar + order_wm
sample_order['all'] = sample_order['human'] + sample_order['151']
df_res = df_res.loc[sample_order['all']]
sns.set(font_scale=0.7)
sns.clustermap(df_res, figsize=(15, 10), cmap="RdBu_r", center=0, vmax=0.3, vmin=-0.3, metric='cosine', annot_kws={"size": 2}, row_cluster=False)

In [ ]:
from sklearn.cluster import SpectralClustering
from sklearn.metrics.pairwise import pairwise_kernels
aff_matrix = pairwise_kernels(df_res.T, metric='cosine') + 2
n_clusters = 8
clustering = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', random_state=4)
clustering.fit(aff_matrix)
cl_order = {i: a for i, a in enumerate([0, 1, 2, 3, 4, 5, 6, 7])}
labels = pd.Series(clustering.labels_, index=df_res.columns).map(cl_order).sort_values()
order = labels.sort_values().index
sns.set(font_scale=0.7)
fig, ax = plt.subplots(figsize=(12, 8))
y = sns.heatmap(df_res.loc[sample_order['all'], order], ax=ax, cmap="RdBu_r", vmax=0.3, vmin=-0.3, center=0)

In [ ]:
layers = ['L1', 'L2', 'L3', 'L4', 'L5', "L6", 'WM']
cluster_color = ['red', 'blue', 'green',  'yellow', 'orange', 'purple', 'olive', 'pink', 'brown', 'black']
clusters_labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']
from matplotlib.patches import bbox_artist
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib import ticker

clusters = np.arange(n_clusters)
colors = cluster_color[:n_clusters]
fig, axes = plt.subplots(7, 1, figsize=(12, 4 * n_clusters), gridspec_kw={'hspace': 0.2})

for ax, layer in zip(axes.flatten(), ['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'WM']):
    df_layer = df_res.loc[df_res.index.str.contains(layer)]
    sns.heatmap(df_layer.loc[:, order], ax=ax, cmap="RdBu_r", vmax=0.3, vmin=-0.3, center=0, xticklabels=False)
    # colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)
    cmap = mpl.colors.ListedColormap(colors)

    cluster_size = labels.value_counts().loc[clusters].values
    cluster_pos = cluster_size.cumsum()
    bounds = [0] + list(cluster_pos)
    norm = mpl.colors.BoundaryNorm(bounds, cmap.N)
    fig.colorbar(
        mpl.cm.ScalarMappable(cmap=cmap, norm=norm),
        cax=cax,
        ticks=bounds,
        orientation='horizontal',
        spacing='proportional'
    )
    cax.xaxis.set_major_locator(ticker.FixedLocator(cluster_pos - cluster_size / 2))
    cax.xaxis.set_major_formatter(ticker.FixedFormatter(clusters))
    cax.xaxis.tick_top()

In [ ]:
df_annotation = df_res.copy()
df_annotation['layer'] = pb_age_norm.obs.layer
df_annotation['condition'] = pb_age_norm.obs.condition
layer_mean = df_annotation.groupby(['condition', 'layer']).mean()
layer_mean.head()
fig, axes = plt.subplots(n_clusters // 2, 2,  figsize=(14, 3.5 * n_clusters // 2), gridspec_kw={'hspace': 0.4})

for label, ax in zip(clusters, axes.flatten()):
    genes = labels[labels == label].index.tolist()

    (
        layer_mean[genes]
        .mean(axis=1)
        .reorder_levels(['layer', 'condition'])
        .unstack().loc[layers]
        .plot
        .line(color={'human': 'tab:purple', 'spatial_libd_human': 'tab:orange'}, ax=ax, marker='.')
    )
    ax.set_title(f'Cluster {label}', fontsize=16)
    ax.grid(False)
    ax.hlines(0, xmin=0, xmax=6, colors='gray', linestyles='dashed')

In [ ]:
labels = pd.DataFrame(labels)
labels = labels.rename(columns={0: "cluster"})
labels

In [ ]:
cluster_id = 1
enr = gseapy.enrichr(gene_list = list(labels[labels['cluster'] == cluster_id].index),
                     background = edge_r.gene,
                     gene_sets=['GO_Biological_Process_2023',
                                'GO_Cellular_Component_2023',
                                'GO_Molecular_Function_2023'],
                     organism='human',
                     outdir=None)
enr_res = enr.results
enr_res = enr_res[enr_res['Adjusted P-value'] < 0.05]
dotplot(enr_res,
              column="Adjusted P-value",
              x='Gene_set',
              size=2,
              top_term=10,
              figsize=(12,15),
              title = "GO analysis",
              xticklabels_rot=10,
              show_ring=True,
              marker='o',
             )

In [ ]:
cluster_id = 1
enr = gseapy.enrichr(gene_list = list(labels[labels['cluster'] == cluster_id].index),
                     background = edge_r.gene,
                     gene_sets=['KEGG_2021_Human',
                                'Reactome_2022'],
                     organism='human',
                     outdir=None)
enr_res = enr.results
enr_res = enr_res[enr_res['Adjusted P-value'] < 0.05]
dotplot(enr_res,
              column="Adjusted P-value",
              x='Gene_set',
              size=2,
              top_term=10,
              figsize=(10,15),
              title = "GO analysis",
              xticklabels_rot=10,
              show_ring=True,
              marker='o',
             )

In [ ]:
labels.to_csv(dir_path + 'genes_by_cluster.csv')

In [ ]:
labels = labels.rename_axis('gene')

cluster_dict = {f"cluster{i}": [] for i in range(8)}

# Iterate over each row of the DataFrame and populate the dictionary
for gene, cluster in labels.iterrows():
    cluster_dict[f"cluster{cluster['cluster']}"].append(gene)

# Print the resulting dictionary
print(cluster_dict)

In [ ]:
adata = sc.read_h5ad(dir_path + 'Velmeshev/velmeshev_snRNA_seq.h5ad')
adata

#Velmeshev data

##Load data

In [ ]:
adata = sc.read_h5ad(dir_path + 'Velmeshev/Copy of velm_pb_mean_ver2.h5ad')
adata_fc = adata[adata.obs.age_range != '1-2 years']
adata_fc = adata_fc[adata_fc.obs.age_range != 'Adult']
adata_fc = adata_fc[adata_fc.obs.age_range != '10-20 years']
adata_fc = adata_fc[adata_fc.obs.age_range != '2-4 years']
adata_fc = adata_fc[adata_fc.obs.age_range != '4-10 years']
adata_fc = adata_fc[adata_fc.obs.region_broad == 'FC']
adata_fc.obs

In [ ]:
sc.pp.filter_genes(adata_fc, min_counts=10)
sc.pp.filter_genes(adata_fc, min_cells=10)

In [ ]:
sc.pp.normalize_total(adata_fc, target_sum=1e4)
sc.pp.log1p(adata_fc)

In [ ]:
adata_fc.obs.lineage

In [ ]:
sc.pp.neighbors(adata_fc)

In [ ]:
sc.tl.umap(adata_fc)

In [ ]:
sc.pl.umap(
    adata_fc,
    color="lineage",
    # Setting a smaller point size to get prevent overlap
    size=5
)


In [ ]:
adata_fc = adata_fc[adata_fc.obs.age_range != '1-2 years']

In [ ]:
adata_fc = adata[adata.obs.region_broad == 'FC']


##T-test (finding cell type markers)

In [ ]:
sc.tl.rank_genes_groups(adata_fc, 'lineage', method='t-test', reference='rest', key_added="t-test", pts=True)
sc.pl.rank_genes_groups(adata_fc, n_genes=20, sharey=False, key="t-test")

In [ ]:
sc.tl.rank_genes_groups(adata_fc, 'lineage', method='logreg', reference='rest', key_added="logreg", pts=True)
sc.pl.rank_genes_groups(adata_fc, n_genes=20, sharey=False, key="logreg")

##separation of genes by p-value

In [ ]:
AST = sc.get.rank_genes_groups_df(adata_fc, group='AST', key='t-test')
ExNeu = sc.get.rank_genes_groups_df(adata_fc, group='ExNeu', key='t-test')
GLIALPROG = sc.get.rank_genes_groups_df(adata_fc, group='GLIALPROG', key='t-test')
IN = sc.get.rank_genes_groups_df(adata_fc, group='IN', key='t-test')
MG = sc.get.rank_genes_groups_df(adata_fc, group='MG', key='t-test')
OL = sc.get.rank_genes_groups_df(adata_fc, group='OL', key='t-test')
OPC = sc.get.rank_genes_groups_df(adata_fc, group='OPC', key='t-test')
VASC = sc.get.rank_genes_groups_df(adata_fc, group='VASC', key='t-test')
OUT = sc.get.rank_genes_groups_df(adata_fc, group='OUT', key='t-test')

In [ ]:
AST_sig = AST[AST['pvals_adj'] < 0.05]
ExNeu_sig = ExNeu[ExNeu['pvals_adj'] < 0.05]
GLIALPROG_sig = GLIALPROG[GLIALPROG['pvals_adj'] < 0.05]
IN_sig = IN[IN['pvals_adj'] < 0.05]
MG_sig = MG[MG['pvals_adj'] < 0.05]
OL_sig = OL[OL['pvals_adj'] < 0.05]
OPC_sig = OPC[OPC['pvals_adj'] < 0.05]
VASC_sig = VASC[VASC['pvals_adj'] < 0.05]
OUT_sig = OUT[OUT['pvals_adj'] < 0.05]


In [ ]:
Neu = pd.concat([ExNeu_sig, IN_sig, OUT_sig])
Neu = Neu.drop_duplicates(subset = 'names')
Neu

In [ ]:
Glia = pd.concat([AST_sig, MG_sig, OL_sig, OPC_sig, GLIALPROG_sig])
Glia = Glia.drop_duplicates(subset = 'names')
Glia

In [ ]:
Neu_sp = Neu[~Neu.names.isin(Glia.names)]
Neu_sp

In [ ]:
Glia_sp = Glia[~Glia.names.isin(Neu.names)]
Glia_sp

In [ ]:
Glia_neu = Glia[Glia.names.isin(Neu.names)]
Glia_neu

In [ ]:
intersect_glia = df_res_7[df_res_7.index.isin(Glia_sp.names)]
intersect_neu = df_res_7[df_res_7.index.isin(Neu_sp.names)]
intersect_all = df_res_7[df_res_7.index.isin(Glia_neu.names)]

In [ ]:
intersect_glia

In [ ]:
from matplotlib.lines import Line2D

sample_order = dict()
for cond in ['human', '151']:
    samples = df_res_7.loc[df_res_7.index.str.contains(cond)].index
    order = samples.sort_values()
    order_laminar = order[order.str.contains('L')].tolist()
    order_wm = order[order.str.contains('WM')].tolist()
    sample_order[cond] = order_wm + order_laminar
sample_order['all'] = sample_order['human'] + sample_order['151']
df_res_7 = df_res_7.loc[sample_order['all']]
sns.set(font_scale=0.7)
cluster = sns.clustermap(df_res_7, figsize=(15, 10), cmap="RdBu_r", center=0, vmax=0.3, vmin=-0.3, metric='cosine', annot_kws={"size": 2}, row_cluster=False)
for i, gene in enumerate(df_res_7.columns):
    if gene in intersect_neu.index:
        cluster.ax_heatmap.axvline(x=i, color='green', linewidth=3, ymin=0, ymax=0.05)
    if gene in intersect_glia.index:
        cluster.ax_heatmap.axvline(x=i, color='red', linewidth=3, ymin=0, ymax=0.05)


# Create custom legend elements
legend_elements = [
    Line2D([0], [0], color='green', lw=3, label='Neu'),
    Line2D([0], [0], color='red', lw=3, label='Glia'),
    Line2D([0], [0], color='blue', lw=3, label='common')
]

# Add legend
cluster.ax_heatmap.legend(handles=legend_elements, loc='upper left')

plt.show()

##separation of genes by score

In [ ]:
AST = sc.get.rank_genes_groups_df(adata_fc, group='AST', key='t-test')
ExNeu = sc.get.rank_genes_groups_df(adata_fc, group='ExNeu', key='t-test')
GLIALPROG = sc.get.rank_genes_groups_df(adata_fc, group='GLIALPROG', key='t-test')
IN = sc.get.rank_genes_groups_df(adata_fc, group='IN', key='t-test')
MG = sc.get.rank_genes_groups_df(adata_fc, group='MG', key='t-test')
OL = sc.get.rank_genes_groups_df(adata_fc, group='OL', key='t-test')
OPC = sc.get.rank_genes_groups_df(adata_fc, group='OPC', key='t-test')
VASC = sc.get.rank_genes_groups_df(adata_fc, group='VASC', key='t-test')
OUT = sc.get.rank_genes_groups_df(adata_fc, group='OUT', key='t-test')
AST = AST.drop(AST.columns[2:8], axis =1)
ExNeu = ExNeu.drop(ExNeu.columns[2:8], axis =1)
GLIALPROG = GLIALPROG.drop(GLIALPROG.columns[2:8], axis =1)
IN = IN.drop(IN.columns[2:8], axis =1)
MG = MG.drop(MG.columns[2:8], axis =1)
OL = OL.drop(OL.columns[2:8], axis =1)
OPC = OPC.drop(OPC.columns[2:8], axis =1)
VASC = VASC.drop(VASC.columns[2:8], axis =1)
OUT = OUT.drop(OUT.columns[2:8], axis =1)

AST.index = AST.names
AST = AST.drop(AST.columns[0], axis = 1)
ExNeu.index = ExNeu.names
ExNeu = ExNeu.drop(ExNeu.columns[0], axis = 1)
GLIALPROG.index = GLIALPROG.names
GLIALPROG = GLIALPROG.drop(GLIALPROG.columns[0], axis = 1)
IN.index = IN.names
IN = IN.drop(IN.columns[0], axis = 1)
MG.index = MG.names
MG = MG.drop(MG.columns[0], axis = 1)
OL.index = OL.names
OL = OL.drop(OL.columns[0], axis = 1)
OPC.index = OPC.names
OPC = OPC.drop(OPC.columns[0], axis = 1)
VASC.index = VASC.names
VASC = VASC.drop(VASC.columns[0], axis = 1)
OUT.index = OUT.names
OUT = OUT.drop(OUT.columns[0], axis = 1)

AST['rank_ast'] = AST['scores'].rank(ascending = False)
ExNeu['rank_exneu'] = ExNeu['scores'].rank(ascending = False)
GLIALPROG['rank_glialprog'] = GLIALPROG['scores'].rank(ascending = False)
IN['rank_in'] = IN['scores'].rank(ascending = False)
MG['rank_mg'] = MG['scores'].rank(ascending = False)
OL['rank_ol'] = OL['scores'].rank(ascending = False)
OPC['rank_opc'] = OPC['scores'].rank(ascending = False)
OUT['rank_out'] = OUT['scores'].rank(ascending = False)
VASC['rank_vasc'] = VASC['scores'].rank(ascending = False)
VASC

In [ ]:
ExNeu

In [ ]:
ranked_genes =  pd.concat([VASC, OUT, AST, ExNeu, IN, GLIALPROG, MG, OPC, OL], axis=1, join="inner")
ranked_genes = ranked_genes.drop(['scores'], axis = 1)
min_ranks = ranked_genes.min(axis=1)
min_genes = ranked_genes.idxmin(axis=1)
min_rank_df = pd.DataFrame({'Gene': min_genes, 'Minimum_Rank': min_ranks})
min_rank_df

In [ ]:
glia = min_rank_df[(min_rank_df['Gene'] == 'rank_ast')|(min_rank_df['Gene'] == 'rank_glialprog')|(min_rank_df['Gene'] == 'rank_mg')|(min_rank_df['Gene'] == 'rank_ol')|(min_rank_df['Gene'] == 'rank_opc')]
glia

In [ ]:
neuron = min_rank_df[(min_rank_df['Gene'] == 'rank_exneu')|(min_rank_df['Gene'] == 'rank_in')|(min_rank_df['Gene'] == 'rank_out')]
neuron

In [ ]:
other = min_rank_df[(min_rank_df['Gene'] == 'rank_vasc')]
other

In [ ]:
df_res_7

In [ ]:
neu_7 = df_res_7_t[df_res_7_t.index.isin(neuron.index)]
gli_7 = df_res_7_t[df_res_7_t.index.isin(glia.index)]
vasc_7 = df_res_7_t[df_res_7_t.index.isin(other.index)]
neu_7


In [ ]:
from matplotlib.lines import Line2D

sample_order = dict()
for cond in ['human', '151']:
    samples = df_res_7.loc[df_res_7.index.str.contains(cond)].index
    order = samples.sort_values()
    order_laminar = order[order.str.contains('L')].tolist()
    order_wm = order[order.str.contains('WM')].tolist()
    sample_order[cond] = order_wm + order_laminar
sample_order['all'] = sample_order['human'] + sample_order['151']
df_res_7 = df_res_7.loc[sample_order['all']]
sns.set(font_scale=0.7)
cluster = sns.clustermap(df_res_7, figsize=(15, 10), cmap="RdBu_r", center=0, vmax=0.3, vmin=-0.3, metric='cosine', annot_kws={"size": 2}, row_cluster=False)
for i, gene in enumerate(df_res_7.columns):
    if gene in neu_7.index:
        cluster.ax_heatmap.axvline(x=i, color='green', linewidth=3, ymin=0, ymax=0.05)
    if gene in gli_7.index:
        cluster.ax_heatmap.axvline(x=i, color='red', linewidth=3, ymin=0, ymax=0.05)
    if gene in vasc_7.index:
        cluster.ax_heatmap.axvline(x=i, color='red', linewidth=3, ymin=0, ymax=0.05)



# Create custom legend elements
legend_elements = [
    Line2D([0], [0], color='green', lw=3, label='Neu'),
    Line2D([0], [0], color='red', lw=3, label='Glia'),
    Line2D([0], [0], color='blue', lw=3, label='common')
]

# Add legend
cluster.ax_heatmap.legend(handles=legend_elements, loc='upper left')

plt.show()

In [ ]:
df_res_7_t = df_res_7.T
df_res_7_t

In [ ]:
OUT

In [ ]:
ranked_genes

In [ ]:
AST_sig = AST[AST['pvals_adj'] < 0.05]
AST_sig

In [ ]:
AST['rank_ast'] = AST['scores'].rank(ascending = False)
AST

In [ ]:
AST = AST.sort_values(by = 'rank_ast', ascending=True)
AST

In [ ]:
ExNeu_sig = ExNeu[ExNeu['pvals_adj'] < 0.05]
ExNeu_sig

In [ ]:
ast_neu = AST_sig[AST_sig.names.isin(ExNeu_sig.names)]
ast_neu

In [ ]:
ast_unique = AST_sig[~AST_sig.names.isin(ExNeu_sig.names)]
ast_unique

In [ ]:
neu_unique = ExNeu_sig[~ExNeu_sig.names.isin(AST_sig.names)]
neu_unique

In [ ]:
neu_unique.names

In [ ]:
df_res_7.columns

In [ ]:
neu_unique = df_res_7_t[df_res_7_t.index.isin(neu_unique.names)]
ast_neu = df_res_7_t[df_res_7_t.index.isin(ast_neu.names)]
ast_neu

In [ ]:
from matplotlib.lines import Line2D

sample_order = dict()
for cond in ['human', '151']:
    samples = df_res_7.loc[df_res_7.index.str.contains(cond)].index
    order = samples.sort_values()
    order_laminar = order[order.str.contains('L')].tolist()
    order_wm = order[order.str.contains('WM')].tolist()
    sample_order[cond] = order_wm + order_laminar
sample_order['all'] = sample_order['human'] + sample_order['151']
df_res_7 = df_res_7.loc[sample_order['all']]
sns.set(font_scale=0.7)
cluster = sns.clustermap(df_res_7, figsize=(15, 10), cmap="RdBu_r", center=0, vmax=0.3, vmin=-0.3, metric='cosine', annot_kws={"size": 2}, row_cluster=False)
for i, gene in enumerate(df_res_7.columns):
    if gene in neu_unique.index:
        cluster.ax_heatmap.axvline(x=i, color='green', linewidth=3, ymin=0, ymax=0.05)
    if gene in ast_unique.index:
        cluster.ax_heatmap.axvline(x=i, color='red', linewidth=3, ymin=0, ymax=0.05)
    if gene in ast_neu.index:
        cluster.ax_heatmap.axvline(x=i, color='blue', linewidth=3, ymin=0, ymax=0.05)

# Create custom legend elements
legend_elements = [
    Line2D([0], [0], color='green', lw=3, label='Neu'),
    Line2D([0], [0], color='red', lw=3, label='Astro'),
    Line2D([0], [0], color='blue', lw=3, label='common')
]

# Add legend
cluster.ax_heatmap.legend(handles=legend_elements, loc='upper right')

plt.show()

In [ ]:
AST

In [ ]:
AST = sc.get.rank_genes_groups_df(adata_fc, group='AST', key='logreg')
AST = AST.sort_values(by = 'scores', ascending = False)
AST = AST.drop(AST.columns[2:7], axis =1)
AST.index = AST.names
AST = AST.drop(AST.columns[0], axis = 1)
pre_res = gp.prerank(rnk= AST, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
AST_res = pre_res.res2d
AST_res['cell_type'] = 'AST'
AST_res

In [ ]:
ExNeu = sc.get.rank_genes_groups_df(adata_fc, group='ExNeu', key='logreg')
ExNeu = ExNeu.sort_values(by = 'scores', ascending = False)
ExNeu = ExNeu.drop(ExNeu.columns[2:7], axis =1)
ExNeu.index = ExNeu.names
ExNeu = ExNeu.drop(ExNeu.columns[0], axis = 1)
pre_res = gp.prerank(rnk= ExNeu, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
ExNeu_res = pre_res.res2d
ExNeu_res['cell_type'] = 'ExNeu'
ExNeu_res

In [ ]:
ExNeu['rank_exneu'] = ExNeu['scores'].rank(ascending = False)
ExNeu

In [ ]:
GLIALPROG = sc.get.rank_genes_groups_df(adata_fc, group='GLIALPROG', key='logreg')
GLIALPROG = GLIALPROG.sort_values(by = 'scores', ascending = False)
GLIALPROG = GLIALPROG.drop(GLIALPROG.columns[2:7], axis =1)
GLIALPROG.index = GLIALPROG.names
GLIALPROG = GLIALPROG.drop(GLIALPROG.columns[0], axis = 1)
pre_res = gp.prerank(rnk= GLIALPROG, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
GLIALPROG_res = pre_res.res2d
GLIALPROG_res['cell_type'] = 'GLIALPROG'
GLIALPROG_res

In [ ]:
GLIALPROG['rank_glialprog'] = GLIALPROG['scores'].rank(ascending = False)
GLIALPROG

In [ ]:
IN = sc.get.rank_genes_groups_df(adata_fc, group='IN', key='logreg')
IN = IN.sort_values(by = 'scores', ascending = False)
IN = IN.drop(IN.columns[2:7], axis =1)
IN.index = IN.names
IN = IN.drop(IN.columns[0], axis = 1)
pre_res = gp.prerank(rnk= IN, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
IN_res = pre_res.res2d
IN_res['cell_type'] = 'IN'
IN_res

In [ ]:
IN['rank_in'] = IN['scores'].rank(ascending = False)
IN

In [ ]:
MG = sc.get.rank_genes_groups_df(adata_fc, group='MG', key='logreg')
MG = MG.sort_values(by = 'scores', ascending = False)
MG = MG.drop(MG.columns[2:7], axis =1)
MG.index = MG.names
MG = MG.drop(MG.columns[0], axis = 1)
pre_res = gp.prerank(rnk= MG, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
MG_res = pre_res.res2d
MG_res['cell_type'] = 'MG'
MG_res

In [ ]:
MG['rank_mg'] = MG['scores'].rank(ascending = False)
MG

In [ ]:
OL = sc.get.rank_genes_groups_df(adata_fc, group='OL', key='logreg')
OL = OL.sort_values(by = 'scores', ascending = False)
OL = OL.drop(OL.columns[2:7], axis =1)
OL.index = OL.names
OL = OL.drop(OL.columns[0], axis = 1)
pre_res = gp.prerank(rnk= OL, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
OL_res = pre_res.res2d
OL_res['cell_type'] = 'OL'
OL_res

In [ ]:
OL['rank_ol'] = OL['scores'].rank(ascending = False)
OL

In [ ]:
OPC = sc.get.rank_genes_groups_df(adata_fc, group='OPC', key='logreg')
OPC = OPC.sort_values(by = 'scores', ascending = False)
OPC = OPC.drop(OPC.columns[2:7], axis =1)
OPC.index = OPC.names
OPC = OPC.drop(OPC.columns[0], axis = 1)
pre_res = gp.prerank(rnk= OPC, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
OPC_res = pre_res.res2d
OPC_res['cell_type'] = 'OPC'
OPC_res

In [ ]:
OPC['rank_opc'] = OPC['scores'].rank(ascending = False)
OPC

In [ ]:
OUT = sc.get.rank_genes_groups_df(adata_fc, group='OUT', key='t-test')


In [ ]:
OUT = sc.get.rank_genes_groups_df(adata_fc, group='OUT', key='logreg')
OUT = OUT.sort_values(by = 'scores', ascending = False)
OUT = OUT.drop(OUT.columns[2:7], axis =1)
OUT.index = OUT.names
OUT = OUT.drop(OUT.columns[0], axis = 1)
pre_res = gp.prerank(rnk= OUT, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
OUT_res = pre_res.res2d
OUT_res['cell_type'] = 'OUT'
OUT_res

In [ ]:
OUT['rank_out'] = OUT['scores'].rank(ascending = False)
OUT

In [ ]:
VASC = sc.get.rank_genes_groups_df(adata_fc, group='VASC', key='logreg')
VASC = VASC.sort_values(by = 'scores', ascending = False)
VASC = VASC.drop(VASC.columns[2:7], axis =1)
VASC.index = VASC.names
VASC = VASC.drop(VASC.columns[0], axis = 1)
pre_res = gp.prerank(rnk= VASC, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
VASC_res = pre_res.res2d
VASC_res['cell_type'] = 'VASC'
VASC_res

In [ ]:
VASC['rank_vasc'] = VASC['scores'].rank(ascending = False)
VASC

In [ ]:
ranked_genes =  pd.concat([VASC, OUT, AST, ExNeu, IN, GLIALPROG, MG, OPC, OL], axis=1, join="inner")
ranked_genes

In [ ]:
ranked_genes = ranked_genes.drop(['scores'], axis = 1)
ranked_genes

In [ ]:
min_ranks = ranked_genes.min(axis=1)
min_ranks

In [ ]:
min_genes = ranked_genes.idxmin(axis=1)
min_genes

In [ ]:
min_genes = ranked_genes.idxmin(axis=1)

min_rank_df = pd.DataFrame({'Gene': min_genes, 'Minimum_Rank': min_ranks})
min_rank_df

In [ ]:
ranked_genes

In [ ]:
ranked_genes =  pd.concat([AST, ExNeu], axis=1, join="inner")
ranked_genes = ranked_genes.drop(['scores'], axis = 1)
min_ranks = ranked_genes.min(axis=1)
min_genes = ranked_genes.idxmin(axis=1)
min_rank_df = pd.DataFrame({'Gene': min_genes, 'Minimum_Rank': min_ranks})
min_rank_df

In [ ]:
astro = min_rank_df[min_rank_df['Gene'] == 'rank_ast']
astro

In [ ]:
neuro = min_rank_df[min_rank_df['Gene'] == 'rank_exneu']
neuro

In [ ]:
FC = pd.read_csv(dir_path + "FC.csv")
FC.columns = FC.columns.str.replace('Unnamed: 0','genes')
FC

In [ ]:
df_res_t = df_res.T
df_res_t['genes'] = df_res_t.index
de_fc = pd.merge(FC, df_res_t, on='genes', how='inner')
de_fc = de_fc.drop(de_fc.iloc[:, 8:119], axis = 1)
de_fc.index = de_fc['genes']
de_fc = de_fc.drop(de_fc.columns[0], axis =1)
de_fc = de_fc.sort_values(by = 'FC_L1', ascending = False)
de_fc = de_fc.dropna(axis = 0)
fig, ax = plt.subplots(figsize= (20, 15))
sns.heatmap(de_fc, cmap = "bwr", ax=ax, norm=mpl.colors.Normalize(vmin=-6, vmax=6))
plt.show()

In [ ]:
cluster_2 = labels.loc[labels.cluster == 2]
cluster_1 = labels.loc[labels.cluster == 1]
cluster_0 = labels.loc[labels.cluster == 0]
cluster_3 = labels.loc[labels.cluster == 3]
cluster_4 = labels.loc[labels.cluster == 4]
cluster_5 = labels.loc[labels.cluster == 5]
cluster_6 = labels.loc[labels.cluster == 6]
cluster_7 = labels.loc[labels.cluster == 7]
de_fc_0 = de_fc.loc[de_fc.index.isin(cluster_0.index)]
de_fc_1 = de_fc.loc[de_fc.index.isin(cluster_1.index)]
de_fc_2 = de_fc.loc[de_fc.index.isin(cluster_2.index)]
de_fc_3 = de_fc.loc[de_fc.index.isin(cluster_3.index)]
de_fc_4 = de_fc.loc[de_fc.index.isin(cluster_4.index)]
de_fc_5 = de_fc.loc[de_fc.index.isin(cluster_5.index)]
de_fc_6 = de_fc.loc[de_fc.index.isin(cluster_6.index)]
de_fc_7 = de_fc.loc[de_fc.index.isin(cluster_7.index)]
de_fc_7

In [ ]:
df = df_res.T
df = df.apply(pd.to_numeric)
df_res_0 = df[df.index.isin(cluster_0.index)]
df_res_1 = df[df.index.isin(cluster_1.index)]
df_res_2 = df[df.index.isin(cluster_2.index)]
df_res_3 = df[df.index.isin(cluster_3.index)]
df_res_4 = df[df.index.isin(cluster_4.index)]
df_res_5 = df[df.index.isin(cluster_5.index)]
df_res_6 = df[df.index.isin(cluster_6.index)]
df_res_7 = df[df.index.isin(cluster_7.index)]

In [ ]:
df_res_7

In [ ]:
cluster_7

In [ ]:
df_res_7

In [ ]:
from matplotlib.lines import Line2D

sample_order = dict()
for cond in ['human', '151']:
    samples = df_res_7.loc[df_res_7.index.str.contains(cond)].index
    order = samples.sort_values()
    order_laminar = order[order.str.contains('L')].tolist()
    order_wm = order[order.str.contains('WM')].tolist()
    sample_order[cond] = order_wm + order_laminar
sample_order['all'] = sample_order['human'] + sample_order['151']
df_res_7 = df_res_7.loc[sample_order['all']]
sns.set(font_scale=0.7)
cluster = sns.clustermap(df_res_7, figsize=(15, 10), cmap="RdBu_r", center=0, vmax=0.3, vmin=-0.3, metric='cosine', annot_kws={"size": 2}, row_cluster=False)
for i, gene in enumerate(df_res_7.columns):
    if gene in intersect_astro.index:
        cluster.ax_heatmap.axvline(x=i, color='green', linewidth=3, ymin=0, ymax=0.05)
    if gene in intersect_neuro.index:
        cluster.ax_heatmap.axvline(x=i, color='red', linewidth=3, ymin=0, ymax=0.05)

# Create custom legend elements
legend_elements = [
    Line2D([0], [0], color='green', lw=3, label='Neu'),
    Line2D([0], [0], color='red', lw=3, label='Astro')
]

# Add legend
cluster.ax_heatmap.legend(handles=legend_elements, loc='upper right')

plt.show()

In [ ]:
neuro

In [ ]:
intersect_astro = df_res_7_t[df_res_7_t.index.isin(astro.index)]
intersect_astro

In [ ]:
intersect_neuro = df_res_7_t[df_res_7_t.index.isin(neuro.index)]
intersect_neuro

In [ ]:
edge_r_full = pd.read_csv(dir_path_er + 'edgeR_age_sampleid.csv')
edge_r_full.columns = edge_r_full.columns.str.replace('Unnamed: 0','gene')
edge_r_full


In [ ]:
import gseapy
from gseapy import barplot, dotplot

In [ ]:
astro.index

##Enrichment for Astro and Neuro genes

In [ ]:
enr = gseapy.enrichr(gene_list = list(neu_7.index),
                     background = edge_r_full.gene,
                     gene_sets=['GO_Biological_Process_2023',
                                'GO_Cellular_Component_2023',
                                'GO_Molecular_Function_2023'],
                     organism='human',
                     outdir=None)
enr_res = enr.results
enr_res = enr_res[enr_res['Adjusted P-value'] < 0.05]
dotplot(enr_res,
              column="Adjusted P-value",
              x='Gene_set',
              size=2,
              top_term=10,
              figsize=(12,15),
              title = "GO analysis",
              xticklabels_rot=10,
              show_ring=True,
              marker='o',
             )

In [ ]:
cluster = sns.clustermap(df, cmap="viridis", figsize=(10, 10))

# Add markings for short gene lists
for i, gene in enumerate(df.index):
    if gene in short_list_1:
        cluster.ax_col_dendrogram.bar(0.5, i, color='green', linewidth=3)
    if gene in short_list_2:
        cluster.ax_col_dendrogram.bar(0.5, i, color='red', linewidth=3)

plt.show()


In [ ]:
df_res_7_t = df_res_7.T

In [ ]:
cell_types = pd.concat([AST_res, ExNeu_res, GLIALPROG_res, IN_res, MG_res, OL_res, OPC_res, OUT_res, VASC_res], axis=0)
pivot_df = cell_types.pivot(index='cell_type', columns='Term', values='NES')
pivot_df = pivot_df.apply(pd.to_numeric)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=False, cmap='bwr', linewidths=0.5, fmt=".2f", center = 0)
plt.title('Enrichment Score Heatmap')
plt.xlabel('Cluster')
plt.ylabel('Cell Type')
plt.show()

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata_fc, n_genes=5, key="logreg", groupby="lineage")

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata_fc, n_genes=5, key="t-test", groupby="lineage", show_gene_labels=True)

#Autism data

In [ ]:
ast_fb = sc.get.rank_genes_groups_df(adata, group='AST-FB', key='logreg')
ast_fb


In [ ]:
ast_fb

In [ ]:
ast_fb = ast_fb.drop(ast_fb.columns[3:7], axis =1)
ast_fb.index = ast_fb.names
ast_fb = ast_fb.drop(ast_fb.columns[0:2], axis = 1)

ast_fb

In [ ]:
ast_fb.index = ast_fb.names
ast_fb = ast_fb.drop(ast_fb.columns[0:2], axis = 1)

In [ ]:
ast_fb

In [ ]:
sc = sc.rename(columns={1: "gene"})
sc

In [ ]:
sc_neuron = sc.loc[sc.Cell == 'Neuron']
sc_neuron

In [ ]:
sc_neuron = sc_neuron.drop(sc_neuron.columns[2:13], axis = 1)

In [ ]:
cluster_dict

In [ ]:
sc_neuron = sc_neuron.drop(sc_neuron.columns[0], axis = 1)

In [ ]:
sc_neuron.index = sc_neuron.Gene

In [ ]:
sc_neuron = sc_neuron.sort_values(by = 'Neuron_log2fc', ascending = False)
sc_neuron

In [ ]:
pre_res = gp.prerank(rnk= ast_fb, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=100, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)

In [ ]:
sc_npc = sc.loc[sc.Cell == 'NPC']
sc_npc

In [ ]:
sc_npc = sc_npc.drop(sc_npc.columns[0], axis = 1)
sc_npc

In [ ]:
sc_npc.index = sc_npc.Gene
sc_npc = sc_npc.sort_values(by = 'IPC_log2fc', ascending = False)
sc_npc

In [ ]:
ast_fb_res = pre_res.res2d
ast_fb_res['cell_type'] = 'AST_FB'
ast_fb_res

In [ ]:
from gseapy import dotplot
# to save your figure, make sure that ``ofname`` is not None
ax = dotplot(pre_res.res2d,
             column="FDR q-val",
             cmap=plt.cm.viridis,
             size=6, # adjust dot size
             figsize=(10,15), cutoff=0.5, show_ring=False)

In [ ]:
adata.obs.cluster

In [ ]:
ast_pp = sc.get.rank_genes_groups_df(adata, group='AST-PP', key='wilcoxon')
ast_pp = ast_pp.drop(ast_pp.columns[2:7], axis =1)
ast_pp.index = ast_pp.names
ast_pp = ast_pp.drop(ast_pp.columns[0], axis = 1)
ast_pp

In [ ]:
pre_res = gp.prerank(rnk= ast_pp, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=1,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
pre_res.res2d
ast_pp_res = pre_res.res2d
ast_pp_res['cell_type'] = 'AST_PP'
ast_pp_res

In [ ]:
from gseapy import dotplot
# to save your figure, make sure that ``ofname`` is not None
ax = dotplot(pre_res.res2d,
             column="FDR q-val",
             cmap=plt.cm.viridis,
             size=6, # adjust dot size
             figsize=(10,15), cutoff=0.25, show_ring=False)

In [ ]:
end

In [ ]:
end = sc.get.rank_genes_groups_df(adata, group='Endothelial', key='wilcoxon')
end = end.drop(end.columns[3:7], axis =1)
end.index = end.names
end = end.drop(end.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= end, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
pre_res.res2d
end_res = pre_res.res2d
end_res['cell_type'] = 'END'
end_res

In [ ]:
in_pv = sc.get.rank_genes_groups_df(adata, group='IN-PV', key='wilcoxon')
in_pv = in_pv.sort_values(by = 'logfoldchanges', ascending = False)
in_pv = in_pv.drop(in_pv.columns[3:7], axis =1)
in_pv.index = in_pv.names
in_pv = in_pv.drop(in_pv.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= in_pv, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
pre_res.res2d
in_pv_res = pre_res.res2d
in_pv_res['cell_type'] = 'IN-PV'
in_pv_res

In [ ]:
for i in

In [ ]:
oligo = sc.get.rank_genes_groups_df(adata, group='Oligodendrocytes', key='wilcoxon')
oligo = oligo.sort_values(by = 'logfoldchanges', ascending = False)
oligo = oligo.drop(oligo.columns[3:7], axis =1)
oligo.index = oligo.names
oligo = oligo.drop(oligo.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= oligo, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
pre_res.res2d
oligo_res = pre_res.res2d
oligo_res['cell_type'] = 'OLIGO'
oligo_res

In [ ]:
opc = sc.get.rank_genes_groups_df(adata, group='OPC', key='wilcoxon')
opc = opc.sort_values(by = 'logfoldchanges', ascending = False)
opc = opc.drop(opc.columns[3:7], axis =1)
opc.index = opc.names
opc = opc.drop(opc.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= opc, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
opc_res = pre_res.res2d
opc_res['cell_type'] = 'OPC'
opc_res

In [ ]:
mg = sc.get.rank_genes_groups_df(adata, group='Microglia', key='wilcoxon')
mg = mg.sort_values(by = 'logfoldchanges', ascending = False)
mg = mg.drop(mg.columns[3:7], axis =1)
mg.index = mg.names
mg = mg.drop(mg.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= mg, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
mg_res = pre_res.res2d
mg_res['cell_type'] = 'Microglia'
mg_res

In [ ]:
in_sst = sc.get.rank_genes_groups_df(adata, group='IN-SST', key='wilcoxon')
in_sst = in_sst.sort_values(by = 'logfoldchanges', ascending = False)
in_sst = in_sst.drop(in_sst.columns[3:7], axis =1)
in_sst.index = in_sst.names
in_sst = in_sst.drop(in_sst.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= in_sst, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_sst_res = pre_res.res2d
in_sst_res['cell_type'] = 'IN-SST'
in_sst_res

In [ ]:
in_sv2c = sc.get.rank_genes_groups_df(adata, group='IN-SV2C', key='wilcoxon')
in_sv2c = in_sv2c.sort_values(by = 'logfoldchanges', ascending = False)
in_sv2c = in_sv2c.drop(in_sv2c.columns[3:7], axis =1)
in_sv2c.index = in_sv2c.names
in_sv2c = in_sv2c.drop(in_sv2c.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= in_sv2c, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_sv2c_res = pre_res.res2d
in_sv2c_res['cell_type'] = 'IN-SV2C'
in_sv2c_res

In [ ]:
in_vip = sc.get.rank_genes_groups_df(adata, group='IN-VIP', key='wilcoxon')
in_vip = in_vip.sort_values(by = 'logfoldchanges', ascending = False)
in_vip = in_vip.drop(in_vip.columns[3:7], axis =1)
in_vip.index = in_vip.names
in_vip = in_vip.drop(in_vip.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= in_vip, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_vip_res = pre_res.res2d
in_vip_res['cell_type'] = 'IN-VIP'
in_vip_res

In [ ]:
l56 = sc.get.rank_genes_groups_df(adata, group='L5/6', key='wilcoxon')
l56 = l56.sort_values(by = 'logfoldchanges', ascending = False)
l56 = l56.drop(l56.columns[3:7], axis =1)
l56.index = l56.names
l56 = l56.drop(l56.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= l56, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l56_res = pre_res.res2d
l56_res['cell_type'] = 'L5/6'
l56_res

In [ ]:
l23 = sc.get.rank_genes_groups_df(adata, group='L2/3', key='wilcoxon')
l23 = l23.sort_values(by = 'logfoldchanges', ascending = False)
l23 = l23.drop(l23.columns[3:7], axis =1)
l23.index = l23.names
l23 = l23.drop(l23.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= l23, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l23_res = pre_res.res2d
l23_res['cell_type'] = 'L2/3'
l23_res

In [ ]:
l4 = sc.get.rank_genes_groups_df(adata, group='L4', key='wilcoxon')
l4 = l4.sort_values(by = 'logfoldchanges', ascending = False)
l4 = l4.drop(l4.columns[3:7], axis =1)
l4.index = l4.names
l4 = l4.drop(l4.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= l4, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l4_res = pre_res.res2d
l4_res['cell_type'] = 'L4'
l4_res

In [ ]:
NeuI = sc.get.rank_genes_groups_df(adata, group='Neu-NRGN-I', key='wilcoxon')
NeuI = NeuI.sort_values(by = 'logfoldchanges', ascending = False)
NeuI = NeuI.drop(NeuI.columns[3:7], axis =1)
NeuI.index = NeuI.names
NeuI = NeuI.drop(NeuI.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= NeuI, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
NeuI_res = pre_res.res2d
NeuI_res['cell_type'] = 'Neu-NRGN-I'
NeuI_res

In [ ]:
NeuII = sc.get.rank_genes_groups_df(adata, group='Neu-NRGN-II', key='wilcoxon')
NeuII = NeuII.sort_values(by = 'logfoldchanges', ascending = False)
NeuII = NeuII.drop(NeuII.columns[3:7], axis =1)
NeuII.index = NeuII.names
NeuII = NeuII.drop(NeuII.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= NeuII, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
NeuII_res = pre_res.res2d
NeuII_res['cell_type'] = 'Neu-NRGN-II'
NeuII_res

In [ ]:
l56cc = sc.get.rank_genes_groups_df(adata, group='L5/6-CC', key='wilcoxon')
l56cc = l56cc.sort_values(by = 'logfoldchanges', ascending = False)
l56cc = l56cc.drop(l56cc.columns[3:7], axis =1)
l56cc.index = l56cc.names
l56cc = l56cc.drop(l56cc.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= l56cc, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l56cc_res = pre_res.res2d
l56cc_res['cell_type'] = 'L5/6-CC'
l56cc_res

In [ ]:
Neu_mat = sc.get.rank_genes_groups_df(adata, group='Neu-mat', key='wilcoxon')
Neu_mat = Neu_mat.sort_values(by = 'logfoldchanges', ascending = False)
Neu_mat = Neu_mat.drop(Neu_mat.columns[3:7], axis =1)
Neu_mat.index = Neu_mat.names
Neu_mat = Neu_mat.drop(Neu_mat.columns[0:2], axis = 1)
pre_res = gp.prerank(rnk= Neu_mat, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
Neu_mat_res = pre_res.res2d
Neu_mat_res['cell_type'] = 'Neu-mat'
Neu_mat_res

In [ ]:
cell_types = pd.concat([ast_fb_res, ast_pp_res, end_res, in_pv_res, oligo_res, opc_res, mg_res, in_sst_res, in_sv2c_res, in_vip_res, l56_res, l23_res, l4_res, NeuI_res, NeuII_res, Neu_mat_res, l56cc_res], axis=0)
cell_types

In [ ]:
sns.clustermap(es_df.astype(float), figsize=(10, 5), cmap='coolwarm', center=0)

In [ ]:
pivot_df = cell_types.pivot(index='cell_type', columns='Term', values='ES')
pivot_df

In [ ]:
pivot_df = pivot_df.apply(pd.to_numeric)
pivot_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=False, cmap='bwr', linewidths=0.5, fmt=".2f", vmin = -1, vmax = 1, center = 0)
plt.title('Enrichment Score Heatmap')
plt.xlabel('Cluster')
plt.ylabel('Cell Type')
plt.show()

In [ ]:
from matplotlib.lines import Line2D
cell_types['dot_size'] = cell_types['FDR q-val'].apply(lambda x: 500 if x < 0.05 else 50)

# Plotting

for cluster in cell_types['Term'].unique():
    plt.figure(figsize=(6, 8))
    cluster_data = cell_types[cell_types['Term'] == cluster]
    sns.scatterplot(data=cluster_data, x='ES', y='cell_type', size='dot_size', legend=False)
    legend_handles = [
        Line2D([0], [0], marker='o', color='blue', markersize=6, label='p-value < 0.05', linewidth=0),
        Line2D([0], [0], marker='o', color='blue', markersize=3, label='p-value ≥ 0.05', linewidth=0)
    ]

    # Add legend with custom handles
    plt.legend(handles=legend_handles, loc='upper left')

    plt.title(f'Enrichment Scores with P-values for {cluster}')
    plt.xlabel('Enrichment Score')
    plt.ylabel('Cell Type')



In [ ]:
sc.tl.rank_genes_groups(adata, 'cluster', method='logreg', reference='rest', key_added="logreg", pts=True)
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, key="logreg")

In [ ]:
ast_fb = sc.get.rank_genes_groups_df(adata, group='AST-FB', key='logreg')
ast_fb = ast_fb.sort_values(by = 'scores', ascending = False)
ast_fb.index = ast_fb.names
ast_fb = ast_fb.drop(ast_fb.columns[0], axis = 1)
pre_res = gp.prerank(rnk= ast_fb, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=100, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
ast_fb_res = pre_res.res2d
ast_fb_res['cell_type'] = 'AST_FB'

ast_pp = sc.get.rank_genes_groups_df(adata, group='AST-PP', key='logreg')
ast_pp = ast_pp.sort_values(by = 'scores', ascending = False)
ast_pp.index = ast_pp.names
ast_pp = ast_pp.drop(ast_pp.columns[0], axis = 1)
pre_res = gp.prerank(rnk= ast_pp, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=1,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
ast_pp_res = pre_res.res2d
ast_pp_res['cell_type'] = 'AST_PP'
end = sc.get.rank_genes_groups_df(adata, group='Endothelial', key='logreg')
end = end.sort_values(by = 'scores', ascending = False)
end.index = end.names
end = end.drop(end.columns[0], axis = 1)
pre_res = gp.prerank(rnk= end, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
end_res = pre_res.res2d
end_res['cell_type'] = 'END'

in_pv = sc.get.rank_genes_groups_df(adata, group='IN-PV', key='logreg')
in_pv = in_pv.sort_values(by = 'scores', ascending = False)
in_pv.index = in_pv.names
in_pv = in_pv.drop(in_pv.columns[0], axis = 1)
pre_res = gp.prerank(rnk= in_pv, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_pv_res = pre_res.res2d
in_pv_res['cell_type'] = 'IN-PV'
oligo = sc.get.rank_genes_groups_df(adata, group='Oligodendrocytes', key='logreg')
oligo = oligo.sort_values(by = 'scores', ascending = False)
oligo.index = oligo.names
oligo = oligo.drop(oligo.columns[0], axis = 1)
pre_res = gp.prerank(rnk= oligo, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
oligo_res = pre_res.res2d
oligo_res['cell_type'] = 'OLIGO'
opc = sc.get.rank_genes_groups_df(adata, group='OPC', key='logreg')
opc = opc.sort_values(by = 'scores', ascending = False)
opc.index = opc.names
opc = opc.drop(opc.columns[0], axis = 1)
pre_res = gp.prerank(rnk= opc, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
opc_res = pre_res.res2d
opc_res['cell_type'] = 'OPC'
mg = sc.get.rank_genes_groups_df(adata, group='Microglia', key='logreg')
mg = mg.sort_values(by = 'scores', ascending = False)
mg.index = mg.names
mg = mg.drop(mg.columns[0], axis = 1)
pre_res = gp.prerank(rnk= mg, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
mg_res = pre_res.res2d
mg_res['cell_type'] = 'Microglia'
in_sst = sc.get.rank_genes_groups_df(adata, group='IN-SST', key='logreg')
in_sst = in_sst.sort_values(by = 'scores', ascending = False)
in_sst.index = in_sst.names
in_sst = in_sst.drop(in_sst.columns[0], axis = 1)
pre_res = gp.prerank(rnk= in_sst, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_sst_res = pre_res.res2d
in_sst_res['cell_type'] = 'IN-SST'
in_sv2c = sc.get.rank_genes_groups_df(adata, group='IN-SV2C', key='logreg')
in_sv2c = in_sv2c.sort_values(by = 'scores', ascending = False)
in_sv2c.index = in_sv2c.names
in_sv2c = in_sv2c.drop(in_sv2c.columns[0], axis = 1)
pre_res = gp.prerank(rnk= in_sv2c, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_sv2c_res = pre_res.res2d
in_sv2c_res['cell_type'] = 'IN-SV2C'
in_vip = sc.get.rank_genes_groups_df(adata, group='IN-VIP', key='logreg')
in_vip = in_vip.sort_values(by = 'scores', ascending = False)
in_vip.index = in_vip.names
in_vip = in_vip.drop(in_vip.columns[0], axis = 1)
pre_res = gp.prerank(rnk= in_vip, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
in_vip_res = pre_res.res2d
in_vip_res['cell_type'] = 'IN-VIP'
l56 = sc.get.rank_genes_groups_df(adata, group='L5/6', key='logreg')
l56 = l56.sort_values(by = 'scores', ascending = False)
l56.index = l56.names
l56 = l56.drop(l56.columns[0], axis = 1)
pre_res = gp.prerank(rnk= l56, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l56_res = pre_res.res2d
l56_res['cell_type'] = 'L5/6'
l23 = sc.get.rank_genes_groups_df(adata, group='L2/3', key='logreg')
l23 = l23.sort_values(by = 'scores', ascending = False)
l23.index = l23.names
l23 = l23.drop(l23.columns[0], axis = 1)
pre_res = gp.prerank(rnk= l23, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l23_res = pre_res.res2d
l23_res['cell_type'] = 'L2/3'
l4 = sc.get.rank_genes_groups_df(adata, group='L4', key='logreg')
l4 = l4.sort_values(by = 'scores', ascending = False)
l4.index = l4.names
l4 = l4.drop(l4.columns[0], axis = 1)
pre_res = gp.prerank(rnk= l4, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l4_res = pre_res.res2d
l4_res['cell_type'] = 'L4'
NeuI = sc.get.rank_genes_groups_df(adata, group='Neu-NRGN-I', key='logreg')
NeuI = NeuI.sort_values(by = 'scores', ascending = False)
NeuI.index = NeuI.names
NeuI = NeuI.drop(NeuI.columns[0], axis = 1)
pre_res = gp.prerank(rnk= NeuI, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
NeuI_res = pre_res.res2d
NeuI_res['cell_type'] = 'Neu-NRGN-I'
NeuII = sc.get.rank_genes_groups_df(adata, group='Neu-NRGN-II', key='logreg')
NeuII = NeuII.sort_values(by = 'scores', ascending = False)
NeuII.index = NeuII.names
NeuII = NeuII.drop(NeuII.columns[0], axis = 1)
pre_res = gp.prerank(rnk= NeuII, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
NeuII_res = pre_res.res2d
NeuII_res['cell_type'] = 'Neu-NRGN-II'
l56cc = sc.get.rank_genes_groups_df(adata, group='L5/6-CC', key='logreg')
l56cc = l56cc.sort_values(by = 'scores', ascending = False)
l56cc.index = l56cc.names
l56cc = l56cc.drop(l56cc.columns[0], axis = 1)
pre_res = gp.prerank(rnk= l56cc, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
l56cc_res = pre_res.res2d
l56cc_res['cell_type'] = 'L5/6-CC'
Neu_mat = sc.get.rank_genes_groups_df(adata, group='Neu-mat', key='logreg')
Neu_mat = Neu_mat.sort_values(by = 'scores', ascending = False)
Neu_mat.index = Neu_mat.names
Neu_mat = Neu_mat.drop(Neu_mat.columns[0], axis = 1)
pre_res = gp.prerank(rnk= Neu_mat, # or rnk = rnk,
                     gene_sets= cluster_dict,
                     threads=4,
                     min_size=4,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True)
Neu_mat_res = pre_res.res2d
Neu_mat_res['cell_type'] = 'Neu-mat'
cell_types = pd.concat([ast_fb_res, ast_pp_res, end_res, in_pv_res, oligo_res, opc_res, mg_res, in_sst_res, in_sv2c_res, in_vip_res, l56_res, l23_res, l4_res, NeuI_res, NeuII_res, Neu_mat_res, l56cc_res], axis=0)
cell_types

In [ ]:
cell_types = pd.concat([ast_fb_res, ast_pp_res, end_res, in_pv_res, oligo_res, opc_res, mg_res, in_sst_res, in_sv2c_res, in_vip_res, l56_res, l23_res, l4_res, NeuI_res, NeuII_res, Neu_mat_res, l56cc_res], axis=0)
pivot_df = cell_types.pivot(index='cell_type', columns='Term', values='ES')
pivot_df = pivot_df.apply(pd.to_numeric)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=False, cmap='bwr', linewidths=0.5, fmt=".2f", vmin = -1, vmax = 1, center = 0)
plt.title('Enrichment Score Heatmap')
plt.xlabel('Cluster')
plt.ylabel('Cell Type')
plt.show()

In [ ]:
pivot_df = cell_types.pivot(index='cell_type', columns='Term', values='NES')
pivot_df = pivot_df.apply(pd.to_numeric)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=False, cmap='bwr', linewidths=0.5, fmt=".2f", center = 0)
plt.title('Enrichment Score Heatmap')
plt.xlabel('Cluster')
plt.ylabel('Cell Type')
plt.show()

In [ ]:
in_pv = sc.get.rank_genes_groups_df(adata, group='IN-PV', key='logreg')
in_pv

In [ ]:
pivot_df['cell_type'] = adata.obs.cluster
pivot_df = pivot_df.sort_values(by = 'cell_type')
pivot_df.index = pivot_df['cell_type']
pivot_df

In [ ]:
pivot_df = pivot_df.drop(pivot_df.columns[8], axis=1)
pivot_df

In [ ]:
  sc_df = pd.DataFrame(adata.X, columns=adata.var_names, index=adata.obs_names)
  sc_df = sc_df.drop(sc_df.columns[16100], axis = 1)
  sc_df = sc_df.T
  sc_df

In [ ]:
ss = gp.ssgsea(data=sc_df,
               gene_sets=cluster_dict,
               outdir=None,
               sample_norm_method='rank',
               verbose = True,
               max_size = 700,
               no_plot=True)

In [ ]:
cluster_dict

In [ ]:
ss_df = ss.res2d
ss_df

In [ ]:
pivot_df = ss_df.pivot(index='Name', columns='Term', values='NES')
pivot_df

In [ ]:
pivot_df = pivot_df.apply(pd.to_numeric)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=False, cmap='bwr', center = 0)
plt.title('Enrichment Score Heatmap')
plt.xlabel('Cluster')
plt.ylabel('Cell Type')
plt.show()

In [ ]:
adata.obs

In [ ]:
df.age